# EduVision_DV - Milestone 1, Module 2
# Data Cleaning & Transformation

**Deliverable:** `education_cleaning.ipynb` (this notebook) + `university_cleaned.csv`

This notebook cleans the 4 raw university-ranking datasets so they are ready for
KPI engineering (Milestone 2) and Tableau dashboards (Milestone 3):

| # | Dataset | Source | Role |
|---|---|---|---|
| 01 | University Overview | QS World University Rankings 2025 | **Master** - every other dataset is synced to this university list |
| 02 | Research Analytics | THE World University Rankings 2024 | Research/citation metrics |
| 03 | Student Analytics | WUR World University Rankings 2023 | Fallback research/citation metrics |
| 04 | Country Comparison | World Bank education indicators, 1970–2016 | Country-level benchmarking |

**What this notebook does, step by step:**
1. Load the 4 raw files and profile them (shape, dtypes, missing values, duplicates).
2. Remove duplicates (none were found, but the check is kept for future re-runs).
3. Standardize university names / country names (verify they already match the master 1:1).
4. Fix a genuine bad category value (`region = 'Not Classified'`) in the master dataset.
5. Sync `02`, `03`, `04` to the master university/country list - drop anything the
   master doesn't recognize.
6. Normalize ranking indicators (numeric rank columns, consistent dtypes).
7. Save Tableau-ready cleaned files and run a final QA report.

> Run the cells in order. Update `RAW_DIR` in the next cell to point at the folder
> containing your 4 raw `.xlsx` files (works both locally and in Google Colab).


## 0. Setup

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

# ---- If running in Google Colab, uncomment the block below to upload files ----
# from google.colab import files
# uploaded = files.upload()   # upload all 4 raw .xlsx files when prompted
# RAW_DIR = "."
# ---------------------------------------------------------------------------

RAW_DIR = "."          # folder containing the 4 raw .xlsx files
OUT_DIR = "cleaned"    # cleaned files will be written here
os.makedirs(OUT_DIR, exist_ok=True)

FILES = {
    "master": "01 University Overview.xlsx",
    "research": "02 Research Analytics.xlsx",
    "student": "03 Student Analytics.xlsx",
    "country": "04 Country Comparison.xlsx",
}

for key, fname in FILES.items():
    path = os.path.join(RAW_DIR, fname)
    assert os.path.exists(path), f"Missing file: {path} - update RAW_DIR or upload the file."

print("All 4 raw files found.")

All 4 raw files found.


## 1. Load & Profile Raw Data

Before touching anything, get an honest picture of each file: row/column counts,
data types, missing values, and duplicate rows. This tells us exactly what
"cleaning" needs to mean for this dataset - in this case the raw files turn out
to be largely clean already at the row level, with two specific problems:

- one bad category label in the master dataset (`region = 'Not Classified'`)
- `04 Country Comparison` containing far more countries than the master
  dataset's university list needs


In [2]:
df_master = pd.read_excel(os.path.join(RAW_DIR, FILES["master"]))
df_research = pd.read_excel(os.path.join(RAW_DIR, FILES["research"]))
df_student = pd.read_excel(os.path.join(RAW_DIR, FILES["student"]))
df_country = pd.read_excel(os.path.join(RAW_DIR, FILES["country"]))

def profile(name, df):
    print(f"--- {name} ---")
    print(f"shape: {df.shape}")
    print(f"missing values: {int(df.isna().sum().sum())}")
    print(f"duplicate rows: {int(df.duplicated().sum())}")
    print()

profile("01 University Overview (master)", df_master)
profile("02 Research Analytics", df_research)
profile("03 Student Analytics", df_student)
profile("04 Country Comparison", df_country)

--- 01 University Overview (master) ---
shape: (1503, 19)
missing values: 0
duplicate rows: 0

--- 02 Research Analytics ---
shape: (867, 27)
missing values: 0
duplicate rows: 0

--- 03 Student Analytics ---
shape: (812, 25)
missing values: 0
duplicate rows: 0

--- 04 Country Comparison ---
shape: (55918, 8)
missing values: 0
duplicate rows: 0



## 2. Remove Duplicates

Check for duplicate rows and duplicate primary keys (`university_id` for 01-03,
`country_id` + `indicator_code` + `year` for 04). This project's raw files had
none, but the check is kept so the notebook is safe to re-run on a refreshed
data pull.


In [3]:
def dedupe(df, subset=None, name=""):
    before = len(df)
    df = df.drop_duplicates(subset=subset, keep="first").reset_index(drop=True)
    removed = before - len(df)
    print(f"{name}: removed {removed} duplicate rows (subset={subset})")
    return df

df_master = dedupe(df_master, subset=["university_id"], name="01 Master")
df_research = dedupe(df_research, subset=["university_id"], name="02 Research")
df_student = dedupe(df_student, subset=["university_id"], name="03 Student")
df_country = dedupe(df_country, subset=["country_id", "indicator_code", "year"], name="04 Country")

01 Master: removed 0 duplicate rows (subset=['university_id'])
02 Research: removed 0 duplicate rows (subset=['university_id'])
03 Student: removed 0 duplicate rows (subset=['university_id'])
04 Country: removed 0 duplicate rows (subset=['country_id', 'indicator_code', 'year'])


## 3. Standardize University & Country Names

The KPI engineering step (Milestone 2) joins these datasets on `university_id`
and `country_id`, so names must line up exactly. Verify - rather than assume -
that `university_name` and `country_id` for the same `university_id` agree
across all three university-level datasets.


In [4]:
def check_name_consistency(df_a, df_b, key, cols, name_a, name_b):
    merged = df_a[[key] + cols].merge(df_b[[key] + cols], on=key, suffixes=("_a", "_b"))
    mismatches = 0
    for c in cols:
        bad = merged[merged[f"{c}_a"] != merged[f"{c}_b"]]
        if len(bad):
            mismatches += len(bad)
            print(f"  Mismatch in '{c}' between {name_a} and {name_b}: {len(bad)} rows")
    if mismatches == 0:
        print(f"  {name_a} vs {name_b}: all checked columns match exactly.")
    return mismatches

print("Checking university_name & country_id consistency:")
check_name_consistency(df_master, df_research, "university_id", ["university_name", "country_id"],
                        "01 Master", "02 Research")
check_name_consistency(df_master, df_student, "university_id", ["university_name", "country_id"],
                        "01 Master", "03 Student")

Checking university_name & country_id consistency:
  01 Master vs 02 Research: all checked columns match exactly.
  01 Master vs 03 Student: all checked columns match exactly.


0

## 4. Fix Bad Category Values in the Master Dataset

One row in the master dataset carries `region = 'Not Classified'`:
**Eastern Mediterranean University**, country_id `CY-N` (Northern Cyprus).

Left as-is, this value gets bucketed separately in every regional visualization
in Tableau. Its sibling country, Cyprus (`CY`), is already classified as
`'Asia'` elsewhere in the same dataset - so we align Northern Cyprus to the
same region rather than inventing a new category or dropping the row.


In [5]:
mask_nc = df_master["region"] == "Not Classified"
print(f"Rows with region = 'Not Classified': {mask_nc.sum()}")
print(df_master.loc[mask_nc, ["university_id", "university_name", "country_id", "region"]])

# Align to the region already used for the sibling country 'CY' (Cyprus)
sibling_region = df_master.loc[df_master["country_id"] == "CY", "region"]
target_region = sibling_region.iloc[0] if len(sibling_region) else "Asia"
df_master.loc[mask_nc, "region"] = target_region

print(f"\nReclassified to region = '{target_region}'.")
assert (df_master["region"] == "Not Classified").sum() == 0, "Not Classified region still present!"


Rows with region = 'Not Classified': 1
    university_id                   university_name country_id          region
616         U0206  Eastern Mediterranean University       CY-N  Not Classified

Reclassified to region = 'Asia'.


## 5. Sync All Datasets to the Master University / Country List

Per project convention, `01 University Overview` (QS 2025) is the **master**
dataset. Every other dataset should only contain universities/countries that
also exist in the master list - this keeps every dashboard filter and every
downstream join fully aligned.

- `02` and `03` are checked against the master's `university_id` list.
- `04` is checked against the master's `country_id` list (it's country-grain,
  not university-grain).


In [6]:
master_university_ids = set(df_master["university_id"])
master_country_ids = set(df_master["country_id"])

def sync_to_master(df, key, master_keys, name):
    before = len(df)
    out = df[df[key].isin(master_keys)].copy()
    removed = before - len(out)
    print(f"{name}: {before} -> {len(out)} rows (removed {removed} not present in master)")
    return out

df_research = sync_to_master(df_research, "university_id", master_university_ids, "02 Research Analytics")
df_student = sync_to_master(df_student, "university_id", master_university_ids, "03 Student Analytics")
df_country = sync_to_master(df_country, "country_id", master_country_ids, "04 Country Comparison")

not_in_wb = master_country_ids - set(df_country["country_id"])
print(f"\nMaster countries with NO World Bank indicator rows available "
      f"(not fabricated, left absent): {sorted(not_in_wb)}")

02 Research Analytics: 867 -> 867 rows (removed 0 not present in master)
03 Student Analytics: 812 -> 812 rows (removed 0 not present in master)
04 Country Comparison: 55918 -> 33450 rows (removed 22468 not present in master)

Master countries with NO World Bank indicator rows available (not fabricated, left absent): ['CY-N', 'TW']


## 6. Normalize Ranking & Numeric Indicators

Make sure numeric columns are actually numeric (not stored as text), and that
categorical/boolean flag columns have a consistent, clean set of values -
required for Tableau to treat them as measures/dimensions correctly.


In [7]:
numeric_cols_master = [
    "academic_reputation", "employer_reputation", "faculty_student_score",
    "citations_per_faculty_score", "international_faculty_score",
    "international_students_score", "international_research_network_score",
    "employment_outcomes_score", "sustainability_score", "overall_score",
]
for c in numeric_cols_master:
    df_master[c] = pd.to_numeric(df_master[c], errors="coerce")

numeric_cols_research = [
    "rank_numeric", "scores_overall", "scores_teaching", "scores_research",
    "scores_citations", "scores_industry_income", "scores_international_outlook",
    "total_students", "student_staff_ratio", "international_students_percentage",
    "female_pct", "male_pct",
]
for c in numeric_cols_research:
    df_research[c] = pd.to_numeric(df_research[c], errors="coerce")

numeric_cols_student = [
    "teaching_score", "research_score", "citations_score", "industry_income_score",
    "international_outlook_score", "total_students", "student_staff_ratio",
    "international_students_percentage", "female_pct", "male_pct", "overall_score",
]
for c in numeric_cols_student:
    df_student[c] = pd.to_numeric(df_student[c], errors="coerce")

df_country["value"] = pd.to_numeric(df_country["value"], errors="coerce")

# Re-check for nulls introduced by a failed numeric coercion (would indicate
# a genuinely dirty value in the source, e.g. text in a numeric column)
for name, df, cols in [
    ("01 Master", df_master, numeric_cols_master),
    ("02 Research", df_research, numeric_cols_research),
    ("03 Student", df_student, numeric_cols_student),
    ("04 Country", df_country, ["value"]),
]:
    n_null = int(df[cols].isna().sum().sum())
    print(f"{name}: {n_null} nulls after numeric coercion (0 expected)")
    assert n_null == 0, f"{name} has values that could not be parsed as numeric - inspect the source."


01 Master: 0 nulls after numeric coercion (0 expected)
02 Research: 0 nulls after numeric coercion (0 expected)
03 Student: 0 nulls after numeric coercion (0 expected)
04 Country: 0 nulls after numeric coercion (0 expected)


## 7. Final QA & Save Cleaned Files

Run a final integrity check on every dataset - no missing values, no duplicate
keys, and every non-master dataset's keys are a subset of the master's keys -
then write the Tableau-ready cleaned files.


In [8]:
def qa(name, df, key_cols):
    missing = int(df.isna().sum().sum())
    dup = int(df.duplicated(subset=key_cols).sum())
    print(f"{name}: shape={df.shape}, missing={missing}, duplicate keys={dup}")
    assert missing == 0, f"{name} still has missing values!"
    assert dup == 0, f"{name} still has duplicate keys!"

qa("01 University Overview (cleaned)", df_master, ["university_id"])
qa("02 Research Analytics (cleaned)", df_research, ["university_id"])
qa("03 Student Analytics (cleaned)", df_student, ["university_id"])
qa("04 Country Comparison (cleaned)", df_country, ["country_id", "indicator_code", "year"])

assert set(df_research["university_id"]).issubset(master_university_ids)
assert set(df_student["university_id"]).issubset(master_university_ids)
assert set(df_country["country_id"]).issubset(master_country_ids)
print("\nAll datasets synced to master and pass QA.")

01 University Overview (cleaned): shape=(1503, 19), missing=0, duplicate keys=0
02 Research Analytics (cleaned): shape=(867, 27), missing=0, duplicate keys=0
03 Student Analytics (cleaned): shape=(812, 25), missing=0, duplicate keys=0


04 Country Comparison (cleaned): shape=(33450, 8), missing=0, duplicate keys=0

All datasets synced to master and pass QA.


In [9]:
# Save cleaned Excel deliverables (Tableau-ready)
df_master.to_excel(os.path.join(OUT_DIR, "01 University Overview.xlsx"), index=False)
df_research.to_excel(os.path.join(OUT_DIR, "02 Research Analytics.xlsx"), index=False)
df_student.to_excel(os.path.join(OUT_DIR, "03 Student Analytics.xlsx"), index=False)
df_country.to_excel(os.path.join(OUT_DIR, "04 Country Comparison.xlsx"), index=False)

# Also save the Milestone-1 deliverable named in the project plan
df_master.to_csv(os.path.join(OUT_DIR, "university_cleaned.csv"), index=False)

print("Saved cleaned files to:", os.path.abspath(OUT_DIR))
for f in sorted(os.listdir(OUT_DIR)):
    print(" -", f)

Saved cleaned files to: /home//work3/testrun/cleaned
 - 01 University Overview.xlsx
 - 02 Research Analytics.xlsx
 - 03 Student Analytics.xlsx
 - 04 Country Comparison.xlsx
 - university_cleaned.csv


## 8. Summary of Changes (Raw → Cleaned)

| Dataset | Rows (raw → cleaned) | What changed |
|---|---|---|
| 01 University Overview | 1,503 → 1,503 | Fixed 1 row: `region` `'Not Classified'` → `'Asia'` (aligned to sibling country Cyprus) |
| 02 Research Analytics | 867 → 867 | No rows removed - already fully synced to master; numeric dtypes enforced |
| 03 Student Analytics | 812 → 812 | No rows removed - already fully synced to master; numeric dtypes enforced |
| 04 Country Comparison | 55,918 → 33,450 | Removed 22,468 rows for 110 countries not present in the master university list |

**Known, documented data gap (not a cleaning bug):** Taiwan (`TW`) and Northern
Cyprus (`CY-N`) exist in the master university list but have no rows in the
World Bank country-indicator file - the World Bank does not track them as
separate economies. This is left as an absence, not filled with fabricated
values.

**Next step (Milestone 2):** these cleaned files feed `generate_education_kpis.py`,
which computes the Six KPIs and builds `university_final_dataset.xlsx` using a
cascading source rule (THE 2024 primary → WUR 2023 fallback) so that every
resulting row has complete data.
